# COVID-19 病例预测（回归）：特征、学习率与网络结构实验

基于李宏毅机器学习课程 ML2023Spring HW1 修改，用全连接神经网络预测每日新增阳性比例（`tested_positive`）。

**与原版 notebook 的主要改动**
- `select_feat`：用训练集上与目标的相关系数绝对值，选出最相关的 `top_k` 个特征
- 所有可调参数（`select_all`、`top_k`、学习率、轮数、TensorBoard 日志名）集中在 `config`
- 同时记录 train / valid loss 到 TensorBoard
- 去掉 Colab 专用代码，可在本地直接运行；补全了预测并保存 `pred.csv` 的代码

实验结果和结论见仓库中的 `README.md`。

代码参考：Heng-Jui Chang @ NTUEE（https://github.com/ga642381/ML2021-Spring/blob/main/HW01/HW01.ipynb）

## 数据

数据来自课程 HW1，**本仓库不包含数据文件**。请从 [Kaggle](https://www.kaggle.com/competitions/ml2023spring-hw1/overview) 下载，并命名为 `covid_train.csv`、`covid_test.csv`，放在本 notebook 同一目录。

## 导入依赖

In [ ]:
import csv
import math
import os

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# 用于记录学习曲线
from torch.utils.tensorboard import SummaryWriter

## 工具函数

In [ ]:
def same_seed(seed):
    '''固定随机种子，保证结果可复现。'''
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_valid_split(data_set, valid_ratio, seed):
    '''把训练数据按比例划分为训练集和验证集。'''
    valid_set_size = int(valid_ratio * len(data_set))
    train_set_size = len(data_set) - valid_set_size
    train_set, valid_set = random_split(
        data_set, [train_set_size, valid_set_size],
        generator=torch.Generator().manual_seed(seed),
    )
    return np.array(train_set), np.array(valid_set)


def predict(test_loader, model, device):
    '''用模型对测试集做预测。'''
    model.eval()  # 评估模式
    preds = []
    for x in tqdm(test_loader):
        x = x.to(device)
        with torch.no_grad():  # 预测时不需要梯度
            pred = model(x)
            preds.append(pred.detach().cpu())
    preds = torch.cat(preds, dim=0).numpy()
    return preds


def save_pred(preds, file):
    '''把预测结果保存为 csv。'''
    with open(file, 'w', newline='') as fp:
        writer = csv.writer(fp)
        writer.writerow(['id', 'tested_positive'])
        for i, p in enumerate(preds):
            writer.writerow([i, p])

## Dataset

In [ ]:
class COVID19Dataset(Dataset):
    '''
    x: 特征
    y: 标签；为 None 时表示测试集（只做预测）
    '''
    def __init__(self, x, y=None):
        if y is None:
            self.y = y
        else:
            self.y = torch.FloatTensor(y)
        self.x = torch.FloatTensor(x)

    def __getitem__(self, idx):
        if self.y is None:
            return self.x[idx]
        return self.x[idx], self.y[idx]

    def __len__(self):
        return len(self.x)

## 模型

三层全连接网络（输入 → 16 → 8 → 1）。实验中也尝试过加宽（64-32-1）和加深（64-32-16-1），验证集 loss 没有提升，详见 README。

In [ ]:
class My_Model(nn.Module):
    def __init__(self, input_dim):
        super(My_Model, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
        )

    def forward(self, x):
        x = self.layers(x)
        x = x.squeeze(1)  # (B, 1) -> (B)
        return x

## 特征选择

用**训练集**上每个特征与目标的相关系数绝对值打分，取分数最高的 `top_k` 个特征。只用训练集计算，避免验证集信息泄露。

In [ ]:
def select_feat(train_data, valid_data, test_data, select_all=True, top_k=10):
    '''选择用于回归的特征。select_all=True 使用全部特征，否则取相关性最高的 top_k 个。'''
    y_train = train_data[:, -1]
    y_valid = valid_data[:, -1]

    raw_x_train = train_data[:, :-1]
    raw_x_valid = valid_data[:, :-1]
    raw_x_test = test_data

    if select_all:
        feat_idx = list(range(raw_x_train.shape[1]))
    else:
        if not 1 <= top_k <= raw_x_train.shape[1]:
            raise ValueError('top_k 必须在 1 和特征总数之间')

        scores = []
        for i in range(raw_x_train.shape[1]):
            feature = raw_x_train[:, i]

            mask = np.isfinite(feature) & np.isfinite(y_train)
            x = feature[mask]
            y = y_train[mask]

            if len(x) < 2:
                scores.append(0.0)
                continue

            x_centered = x - np.mean(x)
            y_centered = y - np.mean(y)
            denominator = np.sqrt(np.sum(x_centered ** 2)) * np.sqrt(np.sum(y_centered ** 2))

            if denominator == 0:  # 常数列，相关系数无定义，记为 0
                scores.append(0.0)
            else:
                corr = np.sum(x_centered * y_centered) / denominator
                scores.append(abs(corr))

        scores = np.array(scores)
        feat_idx = np.argsort(-scores, kind='stable')[:top_k]

        print('选中的特征列：', feat_idx.tolist(), flush=True)
        print('相关性绝对值：', scores[feat_idx], flush=True)

    return (
        raw_x_train[:, feat_idx],
        raw_x_valid[:, feat_idx],
        raw_x_test[:, feat_idx],
        y_train,
        y_valid,
    )

## 训练循环

每个 epoch：训练模式下遍历 batch（清零梯度 → 前向 → 计算 loss → 反向传播 → 更新参数）；然后在验证集上评估（`eval` + `no_grad`），验证 loss 更低就保存模型；连续 `early_stop` 轮没有改进就提前停止。

In [ ]:
def trainer(train_loader, valid_loader, model, config, device):
    criterion = nn.MSELoss(reduction='mean')
    optimizer = torch.optim.SGD(model.parameters(), lr=config['learning_rate'], momentum=0.7)

    # TensorBoard 日志，每次实验请在 config['run_name'] 里换一个名字
    writer = SummaryWriter(log_dir=os.path.join('./runs', config['run_name']))

    os.makedirs(os.path.dirname(config['save_path']), exist_ok=True)

    n_epochs, best_loss, step, early_stop_count = config['n_epochs'], math.inf, 0, 0

    for epoch in range(n_epochs):
        model.train()
        loss_record = []

        train_pbar = tqdm(train_loader, position=0, leave=True)
        for x, y in train_pbar:
            optimizer.zero_grad()               # 梯度清零（PyTorch 默认累加梯度）
            x, y = x.to(device), y.to(device)
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()                     # 反向传播，计算梯度
            optimizer.step()                    # 更新参数
            step += 1
            loss_record.append(loss.detach().item())

            train_pbar.set_description(f'Epoch [{epoch+1}/{n_epochs}]')
            train_pbar.set_postfix({'loss': loss.detach().item()})

        mean_train_loss = sum(loss_record) / len(loss_record)
        writer.add_scalar('Loss/train', mean_train_loss, step)

        model.eval()
        loss_record = []
        for x, y in valid_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                pred = model(x)
                loss = criterion(pred, y)
            loss_record.append(loss.item())

        mean_valid_loss = sum(loss_record) / len(loss_record)
        print(f'Epoch [{epoch+1}/{n_epochs}]: Train loss: {mean_train_loss:.4f}, Valid loss: {mean_valid_loss:.4f}')
        writer.add_scalar('Loss/valid', mean_valid_loss, step)

        if mean_valid_loss < best_loss:
            best_loss = mean_valid_loss
            torch.save(model.state_dict(), config['save_path'])  # 保存验证集上最好的模型
            print('Saving model with loss {:.3f}...'.format(best_loss))
            early_stop_count = 0
        else:
            early_stop_count += 1

        if early_stop_count >= config['early_stop']:
            print('\nModel is not improving, so we halt the training session.')
            break

    writer.close()

## 配置

所有实验只需要修改这里。做对比实验时，**每次只改一个参数**，并给 `run_name` 起一个新名字。

| 实验 | 修改的参数 |
|---|---|
| 全部特征 | `select_all=True` |
| 选 Top-k 特征 | `select_all=False`，改 `top_k` |
| 学习率 | `learning_rate` |
| 训练轮数 | `n_epochs` |

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

config = {
    'seed': 5201314,
    'select_all': False,        # True: 使用全部特征；False: 使用相关性最高的 top_k 个特征
    'top_k': 10,
    'valid_ratio': 0.2,         # 验证集大小 = 训练数据 * valid_ratio
    'n_epochs': 3000,
    'batch_size': 256,
    'learning_rate': 3e-4,
    'early_stop': 600,          # 验证 loss 连续这么多轮没有改进就停止
    'save_path': './models/model.ckpt',
    'run_name': 'topk10_lr3e-4',  # TensorBoard 日志名，每次实验换一个
}

## 读取数据并构建 DataLoader

In [ ]:
same_seed(config['seed'])

train_data = pd.read_csv('./covid_train.csv').values
test_data = pd.read_csv('./covid_test.csv').values

train_data, valid_data = train_valid_split(train_data, config['valid_ratio'], config['seed'])

print(f'train_data size: {train_data.shape}')
print(f'valid_data size: {valid_data.shape}')
print(f'test_data size: {test_data.shape}')

x_train, x_valid, x_test, y_train, y_valid = select_feat(
    train_data, valid_data, test_data,
    select_all=config['select_all'],
    top_k=config['top_k'],
)
print(f'number of features: {x_train.shape[1]}')

train_dataset = COVID19Dataset(x_train, y_train)
valid_dataset = COVID19Dataset(x_valid, y_valid)
test_dataset = COVID19Dataset(x_test)

# num_workers=0：在 Windows 上更稳定
train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=0)

## 开始训练

In [ ]:
model = My_Model(input_dim=x_train.shape[1]).to(device)
trainer(train_loader, valid_loader, model, config, device)

## 查看学习曲线（可选）

在终端运行下面的命令，再用浏览器打开 http://localhost:6006/ ：

```
python -m tensorboard.main --logdir=./runs
```

## 预测

加载验证集上最好的模型，对测试集预测，结果保存到 `pred.csv`。

In [ ]:
model = My_Model(input_dim=x_train.shape[1]).to(device)
model.load_state_dict(torch.load(config['save_path'], map_location=device))
preds = predict(test_loader, model, device)
save_pred(preds, 'pred.csv')